# Create Bounds Shapefiles

This notebook reads the region from `amazon_data` and the 3 regions from `bioflore_data`, creating shapefiles for their bounds.

In [ ]:
import os
import rasterio
from shapely.geometry import box
import geopandas as gpd
import pandas as pd

In [ ]:
# Define paths
amazon_image_path = '/home/luizluz/Documentos/multi-task-fcn/amazon_data/amazon_input_data/orthoimage/orthoimage.tif'

bioflore_dir = '/home/luizluz/Documentos/multi-task-fcn/bioflore_data/input_data/segmentations/train'
bioflore_images = [
    os.path.join(bioflore_dir, 'Mosaic_BigPlot_03.tif'),
    os.path.join(bioflore_dir, 'Mosaic_BigPlot_07.tif'),
    os.path.join(bioflore_dir, 'Mosaic_BigPlot_11.tif')
]

In [ ]:
def get_bounds_and_crs(tif_path, name):
    with rasterio.open(tif_path) as src:
        bounds = src.bounds
        crs = src.crs
        # Create a box geometry
        geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
        return {'name': name, 'geometry': geom, 'path': tif_path}, crs

## Process Amazon Data

In [ ]:
amazon_data = []
amazon_crs = None

if os.path.exists(amazon_image_path):
    data, crs = get_bounds_and_crs(amazon_image_path, 'Amazon_Region')
    amazon_data.append(data)
    amazon_crs = crs
    print(f"Processed Amazon: {amazon_crs}")
else:
    print("Amazon image not found!")

gdf_amazon = gpd.GeoDataFrame(amazon_data, crs=amazon_crs)

## Process Bioflore Data

In [ ]:
bioflore_data_list = []
bioflore_crs = None

for p in bioflore_images:
    if os.path.exists(p):
        name = os.path.basename(p).replace('.tif', '')
        data, crs = get_bounds_and_crs(p, name)
        bioflore_data_list.append(data)
        if bioflore_crs is None:
            bioflore_crs = crs
        elif bioflore_crs != crs:
            print(f"Warning: CRS mismatch for {name}")
    else:
        print(f"File not found: {p}")

gdf_bioflore = gpd.GeoDataFrame(bioflore_data_list, crs=bioflore_crs)
print(f"Processed Bioflore: {bioflore_crs}")

## Save Individual Shapefiles

In [ ]:
output_dir = '/home/luizluz/Documentos/multi-task-fcn/exploration_notebooks/shapes_output'
os.makedirs(output_dir, exist_ok=True)

amazon_shp_path = os.path.join(output_dir, 'amazon_bounds.shp')
bioflore_shp_path = os.path.join(output_dir, 'bioflore_bounds.shp')

gdf_amazon.to_file(amazon_shp_path)
gdf_bioflore.to_file(bioflore_shp_path)

print(f"Saved {amazon_shp_path}")
print(f"Saved {bioflore_shp_path}")

## Merge and Save Combined Shapefile

To ensure compatibility between regions in different UTM zones (Amazon vs Bioflore/Guyana), we reproject everything to **EPSG:4326 (WGS 84 Object Geographic)** before merging. This guarantees that they share the same CRS in the final GeoDataFrame.

In [ ]:
# Add source column first
gdf_amazon['dataset'] = 'Amazon'
gdf_bioflore['dataset'] = 'Bioflore'

# Reproject to EPSG:4326 for combined output
target_crs = "EPSG:4326"
print(f"Reprojecting all to {target_crs}...")

gdf_amazon_4326 = gdf_amazon.to_crs(target_crs)
gdf_bioflore_4326 = gdf_bioflore.to_crs(target_crs)

gdf_combined = pd.concat([gdf_amazon_4326, gdf_bioflore_4326], ignore_index=True)
# Ensure CRS is preserved/set on result
gdf_combined.set_crs(target_crs, allow_override=True, inplace=True)

combined_shp_path = os.path.join(output_dir, 'all_regions_bounds.shp')
gdf_combined.to_file(combined_shp_path)

print(f"Saved combined shapefile: {combined_shp_path}")
print(f"Final CRS: {gdf_combined.crs}")

In [ ]:
# Verify contents
print(gdf_combined[['name', 'dataset', 'geometry']])
gdf_combined.plot()

## Create Centroids Shapefile

We calculate one centroid for the Amazon region and one centroid for the Bioflore region (using the union of its 3 parts).

In [ ]:
# Calculate centroid for Amazon (single polygon)
amazon_centroid = gdf_amazon_4326.geometry.iloc[0].centroid

# Calculate centroid for Bioflore (multiple polygons -> union -> centroid)
bioflore_centroid = gdf_bioflore_4326.geometry.unary_union.centroid

# Create GeoDataFrame
centroids_data = [
    {'name': 'Amazon_Centroid', 'dataset': 'Amazon', 'geometry': amazon_centroid},
    {'name': 'Bioflore_Centroid', 'dataset': 'Bioflore', 'geometry': bioflore_centroid}
]

gdf_centroids = gpd.GeoDataFrame(centroids_data, crs="EPSG:4326")

centroids_shp_path = os.path.join(output_dir, 'centroids.shp')
gdf_centroids.to_file(centroids_shp_path)

print(f"Saved centroids shapefile: {centroids_shp_path}")
print(gdf_centroids)

In [ ]:
# Optional: Plot centroids on top of bounds
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
gdf_combined.plot(ax=ax, color='lightgrey', edgecolor='black', alpha=0.5)
gdf_centroids.plot(ax=ax, color='red', markersize=50)

plt.title("Regions and Centroids")
plt.show()